In [38]:
import numpy as np
import cv2
from joblib import dump, load
import json

import torch
import ultralytics
from ultralytics import YOLO

In [39]:
import os, sys, codecs, glob
from PIL import Image, ImageDraw

import numpy as np
import pandas as pd
import cv2
import glob

import torch
torch.backends.cudnn.benchmark = False
# torch.backends.cudnn.enabled = False

import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data.dataset import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [5]:
model1 = YOLO("./detect.pt")

In [41]:
lbl = load("lbl.pkl")

In [44]:
len(lbl.classes_)

1307

In [110]:
import timm
import math

class Ai4SNet(nn.Module):
    def __init__(self):
        super(Ai4SNet, self).__init__()    
        self.model = timm.create_model('efficientnet_b0', num_classes=len(lbl.classes_), pretrained=True)
        
    def forward(self, img, labels=None):        
        feat = self.model(img)
        return feat
    
model2 = Ai4SNet()
model2.load_state_dict(torch.load("cls.pt"))
model2.eval()

Ai4SNet(
  (model): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw): Conv2d(3

In [121]:
img_path = "./demo/ZHJWD000871-000001-XIAOJIAO2_30_8.jpg"
img = cv2.imread(img_path)
results = model1.predict(img_path)


image 1/1 /root/autodl-tmp/try-1/submit/demo/ZHJWD000871-000001-XIAOJIAO2_30_8.jpg: 1280x576 5 chars, 13.2ms
Speed: 5.0ms preprocess, 13.2ms inference, 1.2ms postprocess per image at shape (1, 3, 1280, 576)


In [122]:
results[0].boxes.xywh.data.cpu().numpy().astype(int)

array([[ 436, 2120,  173,  182],
       [ 499, 1936,  221,  441],
       [ 773, 1899,  234,  502],
       [ 602, 1536,  402,  233],
       [1350,  651,  239,  283]])

In [123]:
class AI4sDataset(Dataset):
    def __init__(self, img_path, img_boxes, transform):
        self.img_path = img_path
        self.img_boxes = img_boxes
        self.transform = transform

    def __getitem__(self, index):
        img = Image.open(self.img_path).convert('RGB')
        
        x, y, w, h = self.img_boxes[index]

        left = x
        top = y
        right = x + w
        bottom = y + h
        img = img.crop((left, top, right, bottom))
        
        if self.transform is not None:
            img = self.transform(img)
        
        return img

    def __len__(self):
        return len(self.img_boxes)

In [124]:
dataset = AI4sDataset(
    img_path, 
    results[0].boxes.xywh.data.cpu().numpy().astype(int),
    transforms.Compose([
        transforms.Resize((128, 128)),
        # transforms.RandomHorizontalFlip(),
        # transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
)

In [125]:
for i in range(len(dataset)):
    label = lbl.inverse_transform([model2(dataset[i].reshape(1, 3, 128, 128))[0].argmax()])[0]
    print(label, json.load(open("ID_to_chinese.json"))[label])

0081 一
0081 一
1013 炎
0081 一
0279 卜


'嗇'

In [89]:
model2(dataset[i].reshape(1, 3, 128, 128))[0]

tensor([ 0.0700,  0.1186,  0.0198,  ..., -0.0018,  0.0184,  0.0592], grad_fn=<SelectBackward0>)